# 02. 오늘의집 상품 카테고리 크롤링

**커널 재시작 시 실행 순서**

1. 셀 1 — driver 상태 확인
2. 셀 2 — driver 죽었을 때만 실행 (새로 생성)
3. 셀 3 — 환경 변수 & 데이터 재로드
4. 셀 4 — 메인 크롤링 실행 (이어서 수집)

## 셀 1. Driver 상태 확인

In [ ]:
try:
    print(driver.current_url)
    print("driver 살아있음")
except Exception as e:
    print("driver 죽었거나 연결 끊김")
    print(e)

## 셀 2. Driver 새로 생성 (죽었을 때만 실행)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--lang=ko-KR")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get("https://store.ohou.se/")
time.sleep(5)

print("새 driver 준비 완료")

## 셀 3. 환경 변수 & 데이터 재로드

In [ ]:
import os
import pandas as pd
import json
import time
from tqdm import tqdm

# 작업 폴더
BASE_DIR = r"C:\Users\SAMSUNG\Desktop\ohouse"

# 파일 경로
cleaned_path = os.path.join(BASE_DIR, "housewarming_cleaned.csv")

# 원본 데이터 다시 불러오기
housewarming_cleaned = pd.read_csv(cleaned_path, encoding="utf-8-sig")

print("BASE_DIR:", BASE_DIR)
print("housewarming_cleaned shape:", housewarming_cleaned.shape)

# 기존 수집 파일 확인
output_path = os.path.join(BASE_DIR, "ohou_product_categories_all_browser_fetch.csv")

if os.path.exists(output_path):
    done_df = pd.read_csv(output_path, encoding="utf-8-sig")
    print("이미 저장된 상품 수:", len(done_df))
else:
    print("아직 저장 파일 없음")

## 셀 4. 메인 크롤링 실행

이미 저장된 productId는 자동으로 건너뛰므로 중단 후 이어서 수집 가능합니다.

In [ ]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from selenium.common.exceptions import WebDriverException

# =========================================================
# 설정
# =========================================================
BUILD_ID = "C8fm57NS2iKwQSGPRZUtn"
CHUNK_SIZE = 50
MAX_DEPTH = 8
MAX_RETRY = 3

output_path = os.path.join(BASE_DIR, "ohou_product_categories_all_browser_fetch.csv")
fail_path = os.path.join(BASE_DIR, "ohou_product_categories_failed_browser_fetch.csv")

print("누적 저장 파일:", output_path)

# =========================================================
# 1. 고유 상품 목록 다시 생성
# =========================================================
product_freq = (
    housewarming_cleaned
    .dropna(subset=["productId"])
    .groupby("productId")
    .size()
    .reset_index(name="tag_count")
)

product_freq["productId"] = product_freq["productId"].astype(int).astype(str)

unique_products_all = (
    housewarming_cleaned
    .dropna(subset=["productId"])
    [["productId", "productName", "brand", "productUrl"]]
    .drop_duplicates(subset=["productId"])
    .copy()
)

unique_products_all["productId"] = unique_products_all["productId"].astype(int).astype(str)

unique_products_all = unique_products_all.merge(
    product_freq,
    on="productId",
    how="left"
)

unique_products_all = unique_products_all.sort_values(
    "tag_count",
    ascending=False
).reset_index(drop=True)

print("전체 고유 상품 수:", len(unique_products_all))

# =========================================================
# 2. 이미 저장된 productId 제외
# =========================================================
if os.path.exists(output_path):
    done_df = pd.read_csv(output_path, encoding="utf-8-sig")
    done_ids = set(done_df["productId"].astype(str))
    print("이미 저장된 상품 수:", len(done_ids))
else:
    done_ids = set()
    print("저장된 파일 없음. 처음부터 시작.")

target_products_resume = unique_products_all[
    ~unique_products_all["productId"].astype(str).isin(done_ids)
].copy()

print("이어서 수집할 상품 수:", len(target_products_resume))

# =========================================================
# 3. 브라우저 도메인 이동
# =========================================================
driver.get("https://store.ohou.se/")
time.sleep(3)
print("driver 준비 완료. 이어서 수집 시작")

# =========================================================
# 4. JSON 안에서 categories 찾기
# =========================================================
def find_categories(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "categories" and isinstance(value, list):
                if len(value) > 0 and isinstance(value[0], dict) and "title" in value[0]:
                    return value

            found = find_categories(value)
            if found is not None:
                return found

    elif isinstance(obj, list):
        for item in obj:
            found = find_categories(item)
            if found is not None:
                return found

    return None

# =========================================================
# 5. 브라우저 내부 fetch 함수
# =========================================================
def browser_fetch_json_batch(urls):
    script = """
    const urls = arguments[0];
    const callback = arguments[arguments.length - 1];

    Promise.all(
        urls.map(url =>
            fetch(url, {
                method: "GET",
                credentials: "include",
                headers: {
                    "accept": "application/json,text/plain,*/*",
                    "x-nextjs-data": "1"
                }
            })
            .then(async response => {
                const text = await response.text();
                return {
                    url: url,
                    status: response.status,
                    ok: response.ok,
                    text: text,
                    error: null
                };
            })
            .catch(error => {
                return {
                    url: url,
                    status: null,
                    ok: false,
                    text: null,
                    error: String(error)
                };
            })
        )
    ).then(results => callback(results));
    """
    return driver.execute_async_script(script, urls)

def browser_fetch_json_batch_safe(urls):
    last_error = None

    for attempt in range(1, MAX_RETRY + 1):
        try:
            return browser_fetch_json_batch(urls)

        except WebDriverException as e:
            last_error = e
            print(f"\nWebDriverException 발생. 재시도 {attempt}/{MAX_RETRY}")
            print(str(e)[:300])

            try:
                driver.get("https://store.ohou.se/")
                time.sleep(5)
            except Exception as refresh_error:
                print("driver refresh 실패:", refresh_error)

            time.sleep(3)

        except Exception as e:
            last_error = e
            print(f"\n알 수 없는 오류. 재시도 {attempt}/{MAX_RETRY}")
            print(str(e)[:300])
            time.sleep(3)

    return [
        {
            "url": url,
            "status": None,
            "ok": False,
            "text": None,
            "error": f"browser_fetch_failed_after_retry: {str(last_error)[:200]}"
        }
        for url in urls
    ]

# =========================================================
# 6. 결과 파싱
# =========================================================
def parse_fetch_result(row, fetch_result):
    product_id = str(row["productId"]).replace(".0", "")

    result = {
        "productId": product_id,
        "productName": row.get("productName", None),
        "brand": row.get("brand", None),
        "productUrl": row.get("productUrl", None),
        "tag_count": row.get("tag_count", None),
        "json_url": fetch_result.get("url"),
        "status_code": fetch_result.get("status"),
        "category_path": None,
        "category_ids": None,
        "category_depth_count": 0,
        "success": False,
        "error": fetch_result.get("error")
    }

    for i in range(1, MAX_DEPTH + 1):
        result[f"category_depth{i}"] = None
        result[f"category_depth{i}_id"] = None

    if fetch_result.get("status") != 200:
        result["error"] = result["error"] or f"status_code_{fetch_result.get('status')}"
        return result

    try:
        data = json.loads(fetch_result.get("text"))
        categories = find_categories(data)

        if not categories:
            result["error"] = "categories_not_found"
            return result

        category_titles = [cat.get("title") for cat in categories]
        category_ids = [cat.get("id") for cat in categories]

        result["category_path"] = " > ".join(category_titles)
        result["category_ids"] = category_ids
        result["category_depth_count"] = len(category_titles)
        result["success"] = True
        result["error"] = None

        for i, title in enumerate(category_titles[:MAX_DEPTH], start=1):
            result[f"category_depth{i}"] = title

        for i, cat_id in enumerate(category_ids[:MAX_DEPTH], start=1):
            result[f"category_depth{i}_id"] = cat_id

        return result

    except Exception as e:
        result["error"] = str(e)
        return result

# =========================================================
# 7. 이어서 수집 실행
# =========================================================
records = target_products_resume.to_dict("records")
total_count = len(records)

start_time = time.time()
run_success = 0
run_fail = 0

for start in tqdm(range(0, total_count, CHUNK_SIZE)):
    chunk = records[start:start + CHUNK_SIZE]

    urls = [
        f"https://store.ohou.se/_next/data/{BUILD_ID}/ko-KR/goods/{str(row['productId']).replace('.0', '')}.json?goodsId={str(row['productId']).replace('.0', '')}"
        for row in chunk
    ]

    fetch_results = browser_fetch_json_batch_safe(urls)

    chunk_results = []
    for row, fetch_result in zip(chunk, fetch_results):
        chunk_results.append(parse_fetch_result(row, fetch_result))

    chunk_df = pd.DataFrame(chunk_results)

    chunk_df.to_csv(
        output_path,
        mode="a",
        header=False,
        index=False,
        encoding="utf-8-sig"
    )

    success_count = chunk_df["success"].sum()
    fail_count = len(chunk_df) - success_count

    run_success += success_count
    run_fail += fail_count

    processed = min(start + CHUNK_SIZE, total_count)

    if processed % 1000 == 0 or processed == total_count:
        elapsed = time.time() - start_time
        speed = processed / elapsed
        remain = total_count - processed
        eta_hour = (remain / speed) / 3600 if speed > 0 else 0

        print(
            f"\n이번 실행 진행: {processed:,}/{total_count:,} | "
            f"이번 실행 성공: {run_success:,} | 실패: {run_fail:,} | "
            f"속도: {speed:.2f}개/sec | "
            f"남은 예상 시간: {eta_hour:.2f}시간"
        )

# =========================================================
# 8. 최종 확인
# =========================================================
category_all = pd.read_csv(output_path, encoding="utf-8-sig")
failed_df = category_all[category_all["success"] == False].copy()

failed_df.to_csv(
    fail_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n========== 현재까지 누적 결과 ==========")
print(f"전체 저장 수: {len(category_all):,}개")
print(f"성공 수: {category_all['success'].sum():,}개")
print(f"실패 수: {(~category_all['success']).sum():,}개")
print(f"성공률: {category_all['success'].mean() * 100:.2f}%")
print(f"최종 파일: {output_path}")
print(f"실패 파일: {fail_path}")

print("\n[status_code 분포]")
print(category_all["status_code"].value_counts(dropna=False).head(20))

print("\n[error 분포]")
print(category_all["error"].value_counts(dropna=False).head(20))

print("\n[category_depth_count 분포]")
print(category_all["category_depth_count"].value_counts(dropna=False).sort_index())

category_all.tail()